# Pathwise Admin Dashboard

**Purpose:** Monitor usage data, user queries, system responses, and basic usage metrics for the Pathwise learning application.

**Authentication:** Password protected (default: `admin123`)

**Sections:**
* Authentication
* Overview Metrics
* Recent Activity
* Usage by Question
* Activity Timeline
* AI Tutor Logs

---

⚠️ **First Time Setup:** Run all cells in order. The table will be created automatically if it doesn't exist.

In [0]:
# Simple password authentication
# Set your password below to authenticate

ADMIN_PASSWORD = "admin123"

# Enter password here:
entered_password = "admin123"  # Change this value to authenticate

print("🔐 Admin Authentication")
print("=" * 40)

if entered_password == ADMIN_PASSWORD:
    print("\n✓ Authentication successful!")
    print("You can now access the admin dashboard.\n")
    authenticated = True
else:
    print("\n✗ Authentication failed. Incorrect password.")
    print("Please update 'entered_password' in the cell and try again.")
    authenticated = False
    raise Exception("Authentication failed. Cannot proceed.")

🔐 Admin Authentication

✓ Authentication successful!
You can now access the [REDACTED] dashboard.



In [0]:
%sql
-- Create the usage_logs table if it doesn't exist
CREATE TABLE IF NOT EXISTS workspace.default.usage_logs (
  log_id STRING NOT NULL,
  timestamp TIMESTAMP NOT NULL,
  user_id STRING NOT NULL,
  session_id STRING NOT NULL,
  event_type STRING NOT NULL,
  question_unit STRING,
  user_input STRING,
  system_response STRING,
  is_correct BOOLEAN,
  response_time_ms INT
)
USING DELTA
COMMENT 'Usage logs for Pathwise learning application';

-- Verify table exists
DESCRIBE EXTENDED workspace.default.usage_logs;

In [0]:
# Check if table has data, if not, generate sample data for testing
from datetime import datetime, timedelta
import uuid
import random

# Check current row count
row_count = spark.sql("SELECT COUNT(*) as cnt FROM workspace.default.usage_logs").collect()[0]['cnt']

print(f"Current rows in usage_logs: {row_count}")

if row_count == 0:
    print("\n⚠️  Table is empty. Generating sample data for demonstration...")
    
    # Sample data
    sample_logs = []
    users = ['user_001', 'user_002', 'user_003', 'user_004']
    sessions = {u: str(uuid.uuid4()) for u in users}
    questions = ['Unit 1.0', 'Unit 1.1', 'Unit 1.2']
    
    base_time = datetime.now() - timedelta(days=7)
    
    for i in range(100):
        user = random.choice(users)
        question = random.choice(questions)
        
        # Question viewed
        sample_logs.append((
            str(uuid.uuid4()),
            base_time + timedelta(hours=i*2, minutes=random.randint(0, 59)),
            user,
            sessions[user],
            'question_viewed',
            question,
            None, None, None, None
        ))
        
        # Answer submitted
        is_correct = random.random() > 0.3
        sample_logs.append((
            str(uuid.uuid4()),
            base_time + timedelta(hours=i*2, minutes=random.randint(0, 59), seconds=30),
            user,
            sessions[user],
            'answer_submitted',
            question,
            f'user_answer_{i}',
            None,
            is_correct,
            random.randint(1000, 5000)
        ))
        
        # Occasional AI query
        if random.random() > 0.7:
            sample_logs.append((
                str(uuid.uuid4()),
                base_time + timedelta(hours=i*2, minutes=random.randint(0, 59), seconds=45),
                user,
                sessions[user],
                'ai_query',
                None,
                f'How do I solve this?',
                None, None, None
            ))
            
            sample_logs.append((
                str(uuid.uuid4()),
                base_time + timedelta(hours=i*2, minutes=random.randint(0, 59), seconds=46),
                user,
                sessions[user],
                'ai_response',
                None,
                f'How do I solve this?',
                'Here is a hint...',
                None,
                random.randint(500, 2000)
            ))
    
    # Create DataFrame
    from pyspark.sql.types import StructType, StructField, StringType, TimestampType, BooleanType, IntegerType
    
    schema = StructType([
        StructField('log_id', StringType(), False),
        StructField('timestamp', TimestampType(), False),
        StructField('user_id', StringType(), False),
        StructField('session_id', StringType(), False),
        StructField('event_type', StringType(), False),
        StructField('question_unit', StringType(), True),
        StructField('user_input', StringType(), True),
        StructField('system_response', StringType(), True),
        StructField('is_correct', BooleanType(), True),
        StructField('response_time_ms', IntegerType(), True)
    ])
    
    df = spark.createDataFrame(sample_logs, schema)
    df.write.mode('append').saveAsTable('workspace.default.usage_logs')
    
    print(f"✓ Generated {len(sample_logs)} sample log entries")
else:
    print("✓ Table already contains data. Skipping sample data generation.")

In [0]:
%sql
-- Key metrics overview
SELECT 
  '📊 Total Events' as metric,
  COUNT(*) as value,
  '' as details
FROM workspace.default.usage_logs

UNION ALL

SELECT 
  '👥 Unique Users' as metric,
  COUNT(DISTINCT user_id) as value,
  '' as details
FROM workspace.default.usage_logs

UNION ALL

SELECT 
  '📝 Answer Submissions' as metric,
  COUNT(*) as value,
  '' as details
FROM workspace.default.usage_logs
WHERE event_type = 'answer_submitted'

UNION ALL

SELECT 
  '✓ Success Rate' as metric,
  ROUND(AVG(CASE WHEN is_correct THEN 100.0 ELSE 0.0 END), 1) as value,
  '%' as details
FROM workspace.default.usage_logs
WHERE event_type = 'answer_submitted'

UNION ALL

SELECT 
  '⏱️ Avg Response Time' as metric,
  ROUND(AVG(response_time_ms), 0) as value,
  'ms' as details
FROM workspace.default.usage_logs
WHERE response_time_ms IS NOT NULL

UNION ALL

SELECT 
  '🤖 AI Queries' as metric,
  COUNT(*) as value,
  '' as details
FROM workspace.default.usage_logs
WHERE event_type = 'ai_query'

ORDER BY metric;

In [0]:
%sql
-- Recent activity log
SELECT 
  DATE_FORMAT(timestamp, 'yyyy-MM-dd HH:mm:ss') as time,
  user_id,
  event_type,
  question_unit,
  CASE 
    WHEN LENGTH(user_input) > 50 THEN CONCAT(SUBSTRING(user_input, 1, 50), '...')
    ELSE user_input 
  END as user_input,
  is_correct,
  response_time_ms
FROM workspace.default.usage_logs
ORDER BY timestamp DESC
LIMIT 50;

In [0]:
%sql
-- Usage breakdown by question
SELECT 
  COALESCE(question_unit, 'N/A (General)') as question_unit,
  COUNT(*) as total_views,
  SUM(CASE WHEN event_type = 'answer_submitted' THEN 1 ELSE 0 END) as submissions,
  SUM(CASE WHEN is_correct = true THEN 1 ELSE 0 END) as correct_answers,
  ROUND(
    100.0 * SUM(CASE WHEN is_correct = true THEN 1 ELSE 0 END) / 
    NULLIF(SUM(CASE WHEN event_type = 'answer_submitted' THEN 1 ELSE 0 END), 0),
    1
  ) as success_rate_pct,
  ROUND(AVG(CASE WHEN event_type = 'answer_submitted' THEN response_time_ms END), 0) as avg_response_ms
FROM workspace.default.usage_logs
GROUP BY question_unit
ORDER BY question_unit;

In [0]:
# Activity timeline visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

# Get activity by hour
df = spark.sql("""
  SELECT 
    DATE_TRUNC('hour', timestamp) as hour,
    event_type,
    COUNT(*) as count
  FROM workspace.default.usage_logs
  GROUP BY DATE_TRUNC('hour', timestamp), event_type
  ORDER BY hour
""").toPandas()

if len(df) > 0:
    # Create pivot table for stacked area chart
    pivot_df = df.pivot(index='hour', columns='event_type', values='count').fillna(0)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, 5), facecolor='#0F1117')
    ax.set_facecolor('#1E2435')
    
    # Plot stacked area
    pivot_df.plot.area(ax=ax, alpha=0.7, 
                       color=['#00C9A7', '#4C8BF5', '#FFB627', '#F87171'],
                       linewidth=0)
    
    # Styling
    ax.set_title('Activity Timeline by Event Type', 
                 fontsize=14, color='#E8ECF4', pad=15, weight='bold')
    ax.set_xlabel('Time', fontsize=11, color='#6B7A99')
    ax.set_ylabel('Event Count', fontsize=11, color='#6B7A99')
    ax.legend(title='Event Type', loc='upper left', 
              facecolor='#161B27', edgecolor='#2E3650',
              labelcolor='#E8ECF4', title_fontsize=10)
    
    # Grid
    ax.grid(True, alpha=0.2, color='#2E3650', linestyle='--')
    ax.spines['top'].set_color('#2E3650')
    ax.spines['right'].set_color('#2E3650')
    ax.spines['bottom'].set_color('#2E3650')
    ax.spines['left'].set_color('#2E3650')
    ax.tick_params(colors='#6B7A99')
    
    # Format x-axis dates
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✓ Displayed timeline for {len(df)} data points")
else:
    print("⚠️ No data available for timeline chart")

In [0]:
%sql
-- AI Tutor interaction logs
SELECT 
  DATE_FORMAT(timestamp, 'yyyy-MM-dd HH:mm:ss') as time,
  user_id,
  CASE 
    WHEN LENGTH(user_input) > 80 THEN CONCAT(SUBSTRING(user_input, 1, 80), '...')
    ELSE user_input 
  END as query,
  CASE 
    WHEN LENGTH(system_response) > 80 THEN CONCAT(SUBSTRING(system_response, 1, 80), '...')
    ELSE system_response 
  END as response,
  response_time_ms
FROM workspace.default.usage_logs
WHERE event_type IN ('ai_query', 'ai_response')
ORDER BY timestamp DESC
LIMIT 50;

---

## 📋 How to Use This Dashboard

1. **Authentication:** Run the authentication cell first (password: `admin123`)
2. **Setup:** Run the table creation cell on first use
3. **View Data:** Run any metric cells to see current statistics
4. **Refresh:** Re-run cells to see updated data as new logs come in

## 🔄 Data Integration

Logs are automatically collected when your Pathwise app uses the `usage_logger` module:

```python
from usage_logger import UsageLogger
logger = UsageLogger()

logger.log_question_view("Unit 1.0")
logger.log_answer_submission("Unit 1.0", "answer", is_correct=True)
logger.log_ai_query("How do I...")
```

See `ADMIN_SETUP.md` for complete integration guide.

## 🔒 Security

* Default password should be changed for production use
* Table contains user interaction data - handle according to privacy policies
* Consider implementing role-based access control